In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q cerebras-cloud-sdk pandas tqdm scikit-learn
!pip install arabert transformers sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.1/106.1 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 9.1 MB/s eta 0:00:00
  Created wheel for emoji: filename=emoji-1.4.2-py3-none-any.whl size=186456 sha256=3ebd8a118e1272d5e1e0c965706b6773ba8aa97760bdb78fa9fb78815c263bdc
  Stored in directory: /root/.cache/pip/wheels/bb/f1/26/f9002669ef6ad80a3c9f1b22880b35d9b4c6650011acee0523
Successfully built emoji


In [ ]:
import os
import re
import json
import zipfile

import pandas as pd
import numpy as np

from tqdm import tqdm

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix
)

from cerebras.cloud.sdk import Cerebras

In [ ]:
# ==========================================
# Configuration
# ==========================================




MODEL_NAME = "gemma-4-31b"

TRAIN_CSV = "/content/drive/MyDrive/University/Models/train_track_2.csv"

EVAL_CSV = "/content/drive/MyDrive/University/Models/test.csv"

# devSet = "/content/drive/MyDrive/University/Models/dev_track_2_ar.csv"
# testSet = "/content/drive/MyDrive/University/Models/test.csv"
# trainSet = "/content/drive/MyDrive/University/Models/train_track_2.csv"

OUTPUT_FILE = "submission.txt"

In [ ]:
# ==========================================
# Batch Configuration
# ==========================================

BATCH_SIZE = 20

TEMPERATURE = 0.0

MAX_TOKENS = 300

In [ ]:
client = Cerebras(
    api_key=API_KEY
)

In [ ]:
# ==========================================
# Load Dataset
# ==========================================

train_df = pd.read_csv(
    TRAIN_CSV,
    keep_default_na=False
)

eval_df = pd.read_csv(
    EVAL_CSV,          # Can be DEV_CSV or TEST_CSV
    keep_default_na=False
)

print("Train Shape :", train_df.shape)
print("Eval Shape  :", eval_df.shape)

print("\nTrain Columns")
print(train_df.columns.tolist())

print("\nEval Columns")
print(eval_df.columns.tolist())

display(train_df.head())
display(eval_df.head())

Train Shape : (2721, 14)
Eval Shape  : (644, 4)

Train Columns
['ID', 'text', 'target', 'stance', 'stance:confidence', 'against_reason', 'favor_reason', 'none_reason', 'sarcasm', 'sarcasm:confidence', 'sentiment', 'sentiment:confidence', 'datetime', 'Date']

Eval Columns
['ID', 'tweet_id', 'text', 'target']


,ID,text,target,stance,stance:confidence,against_reason,favor_reason,none_reason,sarcasm,sarcasm:confidence,sentiment,sentiment:confidence,datetime,Date
0,3,روح حلل محد يم تطعيم كورونا شف الحرم البارح م...,Covid Vaccine,None,0.4003,,,Not clear,Yes,0.5990,Neutral,0.6180,2022-04-28 11:12:56+00:00,28/04/2022
1,6,#LEAP22 مؤتمر يجمع اشهر وابرز المؤثرين في الم...,Digital Transformation,Favor,1.0000,,F_Explicit,,No,1.0000,Positive,0.7531,2022-02-02 18:24:09+00:00,02/02/2022
2,7,خصوصية البيانات وحمايتها في المنظمة مطلب ولكن ...,Digital Transformation,Favor,0.7559,,F_Explicit,,No,1.0000,Neutral,0.8116,2022-03-27 10:36:04+00:00,27/03/2022
3,10,كعادة البشر ذوي العقليات المحدودة والكسولة الك...,Digital Transformation,Favor,1.0000,,F_Explicit,,No,0.8808,Negative,0.6450,2021-05-02 00:32:17+00:00,02/05/2021
4,11,MENTION MENTION MENTION وش فائدة اللقاح بما ان...,Covid Vaccine,Against,0.8233,A_Explicit,,,No,1.0000,Negative,1.0000,2021-06-17 12:43:40+00:00,17/06/2021


,ID,tweet_id,text,target
0,1,1621910933122551812,MENTION شايله هم الفصل الثالث من الحين 💀,Trimester
1,2,1621906921803419650,MENTION طاقة الطالب النفسيه مع ثلاث فصول لاتسم...,Trimester
2,3,1621906857764831233,مع الاترام الثلاثة المفروض اليوم الدراسي قصير ...,Trimester
3,4,1621901035152228361,MENTION لا ، اللي له تأثير ايجابي صدق هو ينلغى...,Trimester
4,5,1621897329828663303,MENTION إلغاء الفصل الدراسي الثالث,Trimester


In [ ]:
# ==========================================
# Verify IDs
# ==========================================

assert train_df["ID"].is_unique, "Duplicate IDs found in train set."

assert eval_df["ID"].is_unique, "Duplicate IDs found in evaluation set."

print("✓ Train IDs are unique.")
print("✓ Evaluation IDs are unique.")

✓ Train IDs are unique.
✓ Evaluation IDs are unique.


In [ ]:
label2id = {
    "Favor": 0,
    "Against": 1,
    "None": 2
}

id2label = {
    0: "Favor",
    1: "Against",
    2: "None"
}

In [ ]:
# ==========================================
# Split Dataset into Batches
# ==========================================

def create_batches(df, batch_size):

    batches = []

    for start in range(0, len(df), batch_size):

        end = min(start + batch_size, len(df))

        # Preserve the original dataframe rows and IDs
        batches.append(
            df.iloc[start:end].copy()
        )

    return batches


# Create batches from the evaluation dataframe
batches = create_batches(

    eval_df,

    BATCH_SIZE

)

print(f"Total batches : {len(batches)}")
print(f"Total tweets  : {len(eval_df)}")

Total batches : 33
Total tweets  : 644


In [ ]:
# ==========================================
# Prompt Variants
# ==========================================

PROMPT_VARIANTS = {

    # ======================================
    # Version 1
    # Concept Understanding
    # ======================================
    "Baseline" : """""",
    "V1": """
1. Distinguish the target's concept from its implementation.
- If the author criticizes how the target is applied, managed, or implemented while implying that the target itself is desirable or necessary, classify as Favor.
- Criticism of execution does not automatically imply opposition to the target.

2. Compare the ideal with reality.
- If the author highlights a failure, injustice, or contradiction to argue that the target should exist or should be implemented better, classify as Favor.
- Negative descriptions of current conditions may express support for the target rather than opposition.

3. Separate sentiment from stance.
- Negative emotions directed toward a person, organization, government, or specific event do not automatically indicate an Against stance toward the target.
- If the target itself is not being evaluated, prefer None.
""",

    # ======================================
    # Version 2
    # + Sarcasm Understanding
    # ======================================
    "V2": """
1. Distinguish the target's concept from its implementation.
- If the author criticizes how the target is applied, managed, or implemented while implying that the target itself is desirable or necessary, classify as Favor.
- Criticism of execution does not automatically imply opposition to the target.

2. Compare the ideal with reality.
- If the author highlights a failure, injustice, or contradiction to argue that the target should exist or should be implemented better, classify as Favor.
- Negative descriptions of current conditions may express support for the target rather than opposition.

3. Separate sentiment from stance.
- Negative emotions directed toward a person, organization, government, or specific event do not automatically indicate an Against stance toward the target.
- If the target itself is not being evaluated, prefer None.

4. Detect sarcastic positive framing.
- Positive words, blessings, compliments, or affectionate expressions may be sarcastic.
- If they appear alongside contradictory facts, criticism, or obvious mockery, interpret the underlying meaning rather than the literal wording.

5. Interpret emoji in context.
- Laughing or crying emojis (😂 🤣 😭 😅) often indicate sarcasm or mockery.
- When emojis contradict the literal wording, prioritize the intended meaning over the surface text.
""",

    # ======================================
    # Version 3
    # Complete Rule Set
    # ======================================
    "V3": """
1. Distinguish the target's concept from its implementation.
- If the author criticizes how the target is applied, managed, or implemented while implying that the target itself is desirable or necessary, classify as Favor.
- Criticism of execution does not automatically imply opposition to the target.

2. Compare the ideal with reality.
- If the author highlights a failure, injustice, or contradiction to argue that the target should exist or should be implemented better, classify as Favor.
- Negative descriptions of current conditions may express support for the target rather than opposition.

3. Separate sentiment from stance.
- Negative emotions directed toward a person, organization, government, or specific event do not automatically indicate an Against stance toward the target.
- If the target itself is not being evaluated, prefer None.

4. Detect sarcastic positive framing.
- Positive words, blessings, compliments, or affectionate expressions may be sarcastic.
- If they appear alongside contradictory facts, criticism, or obvious mockery, interpret the underlying meaning rather than the literal wording.

5. Interpret emoji in context.
- Laughing or crying emojis (😂 🤣 😭 😅) often indicate sarcasm or mockery.
- When emojis contradict the literal wording, prioritize the intended meaning over the surface text.

6. Handle rhetorical questions carefully.
- Rhetorical questions often express implicit criticism.
- Determine whether the criticism targets the target itself or its implementation.
- If the criticism attacks only the implementation, classify as Favor.
- If it attacks the target's underlying idea or logic, classify as Against.

7. Handle exaggeration and hyperbole.
- Hyperbolic or absurd scenarios are frequently used for sarcasm.
- Determine whether the exaggeration mocks the target itself or the current implementation before assigning a label.

8. Do not over-predict None.
- If a reasonable Favor or Against stance can be inferred from implicit meaning, sarcasm, rhetorical language, or context, prefer Favor or Against.
- Use None only when no stance toward the target can reasonably be inferred.

9. Focus only on the author's stance toward the specified target.
- Ignore opinions about associated people, institutions, or events unless they clearly imply a stance toward the target itself.
"""
}

In [ ]:
SYSTEM_PROMPT = """
You are an expert Arabic stance detection system specializing in social media.

Your task is to determine the author's stance toward the specified target.

Possible labels:
- Favor
- Against
- None

Label definitions:

Favor:
The tweet explicitly or implicitly supports, agrees with, praises, promotes, or defends the target.

Against:
The tweet explicitly or implicitly opposes, criticizes, rejects, mocks, attacks, or condemns the target.

None:
The tweet expresses no identifiable stance toward the target.

Guidelines:

- Focus ONLY on the stance toward the given target.
- Distinguish stance from general sentiment.
- Consider implicit opinions.
- Consider sarcasm, irony, rhetorical questions and figurative language.
- Ignore opinions about unrelated people or events.
- Ignore URLs unless they contribute to the stance.
- Consider hashtags only if they express the author's opinion.
- If Favor or Against can reasonably be inferred, prefer them over None.
- Use None only when no stance can be inferred.

Return only valid JSON.
Never explain your reasoning.
"""

In [ ]:
# ==========================================
# Build Batch Prompt
# ==========================================

def build_batch_prompt(batch_df, variant="V1"):

    reasoning_rules = PROMPT_VARIANTS[variant]

    prompt = f"""
You are an expert Arabic stance detection system.

Your task is to determine the stance expressed in each Arabic tweet toward its given target.

Possible labels:
- Favor
- Against
- None

Definitions:

Favor:
The tweet explicitly or implicitly supports, agrees with, praises, promotes, or defends the target.

Against:
The tweet explicitly or implicitly opposes, criticizes, rejects, mocks, attacks, or condemns the target.

None:
The tweet contains no clear stance toward the target.
Choose None ONLY if no reasonable Favor or Against stance can be inferred.

Guidelines:

- Consider implicit opinions.
- Consider sarcasm and irony.
- Consider context.
- Ignore URLs unless they contribute to the stance.
- Ignore hashtags unless they express the author's opinion.
- Focus ONLY on the author's stance toward the given target.

Additional reasoning guidelines:

- Interpret the mention of official initiatives, workshops, or conferences related to the target as implicit support (Favor).
- Identify Favor stances in tweets that advocate for the correct application or a broader understanding of the target.
- Analyze the role of the target in the sentence; if the author presents the target as a goal, a right, or a positive development, label it Favor.
- Be cautious with sarcasm: if a user mocks the absence or failure of the target, the stance toward the target itself is Favor.
- Avoid labeling a tweet as None simply because it lacks emotive adjectives; factual reporting of positive progress can express an implicit stance.
- Differentiate between attacking the target's concept (Against) and attacking those who oppose the target (Favor).

{reasoning_rules}

- Predict exactly ONE label for EVERY tweet.
- Do NOT skip any tweet.
- Do NOT invent labels.
- Use ONLY these labels exactly:
  Favor
  Against
  None

If the tweet clearly expresses an opinion toward the target, choose Favor or Against.
Use None only when the stance is genuinely absent or impossible to infer.

Return ONLY valid JSON.

Example output:

{{
"12345":"Favor",
"12346":"Against",
"12347":"None"
}}

Tweets:

"""

    for _, row in batch_df.iterrows():

        prompt += f"""
ID: {row['ID']}

Target:
{row['target']}

Tweet:
{row['text']}

"""

    prompt += """

Return ONLY the JSON object.

Do NOT explain.
Do NOT use markdown.
Do NOT wrap the JSON inside ```.

Ensure every ID appears exactly once.
Use the same IDs provided above.

"""

    return prompt

In [ ]:
import json
import re
import time

VALID_LABELS = {
    "Favor",
    "Against",
    "None"
}

def predict_batch(batch_df, variant):

    prompt = build_batch_prompt(batch_df, variant)

    # Expected IDs from the CSV
    expected_ids = {
        str(i) for i in batch_df["ID"].tolist()
    }

    while True:

        try:

            response = client.chat.completions.create(

                model=MODEL_NAME,

                temperature=TEMPERATURE,

                max_tokens=MAX_TOKENS,

                messages=[

                    {
                        "role": "system",
                        "content": SYSTEM_PROMPT
                    },

                    {
                        "role": "user",
                        "content": prompt
                    }

                ]

            )

            # ------------------------------------
            # Extract response safely
            # ------------------------------------

            message = response.choices[0].message

            if message.content is None:

                raise ValueError(
                    "Model returned no content."
                )

            text = message.content.strip()

            # ------------------------------------
            # Remove markdown
            # ------------------------------------

            text = re.sub(
                r"```json",
                "",
                text,
                flags=re.IGNORECASE
            )

            text = re.sub(
                r"```",
                "",
                text
            )

            text = text.strip()

            # ------------------------------------
            # Parse JSON
            # ------------------------------------

            parsed = None

            try:

                parsed = json.loads(text)

            except:

                match = re.search(
                    r"\{.*\}",
                    text,
                    re.DOTALL
                )

                if match:

                    parsed = json.loads(
                        match.group()
                    )

            if parsed is None:

                raise ValueError(
                    "Could not parse JSON."
                )

            # ------------------------------------
            # Verify IDs
            # ------------------------------------

            returned_ids = set(parsed.keys())

            if returned_ids != expected_ids:

                print("=" * 80)
                print("ID MISMATCH")
                print("=" * 80)

                print("Expected:", expected_ids)
                print("Returned:", returned_ids)

                raise ValueError(
                    "Returned IDs do not match batch."
                )

            # ------------------------------------
            # Verify labels
            # ------------------------------------

            for tweet_id, label in parsed.items():

                if label not in VALID_LABELS:

                    raise ValueError(
                        f"Invalid label '{label}' for ID {tweet_id}"
                    )

            # ------------------------------------
            # Convert keys to integers
            # ------------------------------------

            parsed = {

                int(k): v

                for k, v in parsed.items()

            }

            return parsed

        except Exception as e:

            print(e)

            print("Retrying in 5 seconds...")

            time.sleep(5)

In [ ]:

# ==========================================
# Run All Prompt Variants
# ==========================================

all_variant_predictions = {}

for variant in ["V2"]:

    print("=" * 80)
    print(f"Running Variant {variant}")
    print("=" * 80)

    predictions = {}

    for batch in tqdm(batches):

        outputs = predict_batch(
            batch,
            variant
        )

        predictions.update(outputs)

    prediction_list = [

        predictions[id_]

        for id_ in eval_df["ID"]

    ]

    all_variant_predictions[variant] = prediction_list


Running Variant Baseline


100%|██████████| 33/33 [06:03<00:00, 11.03s/it]


Running Variant V1


 55%|█████▍    | 18/33 [04:00<02:39, 10.64s/it]

Error code: 429 - {'message': 'Requests per hour limit exceeded - too many requests sent.', 'type': 'too_many_requests_error', 'param': 'quota', 'code': 'request_quota_exceeded'}
Retrying in 5 seconds...
Error code: 429 - {'message': 'Requests per hour limit exceeded - too many requests sent.', 'type': 'too_many_requests_error', 'param': 'quota', 'code': 'request_quota_exceeded'}
Retrying in 5 seconds...
Error code: 429 - {'message': 'Requests per hour limit exceeded - too many requests sent.', 'type': 'too_many_requests_error', 'param': 'quota', 'code': 'request_quota_exceeded'}
Retrying in 5 seconds...


 58%|█████▊    | 19/33 [04:21<03:14, 13.91s/it]

Error code: 429 - {'message': 'Requests per hour limit exceeded - too many requests sent.', 'type': 'too_many_requests_error', 'param': 'quota', 'code': 'request_quota_exceeded'}
Retrying in 5 seconds...


 58%|█████▊    | 19/33 [04:26<03:16, 14.01s/it]


KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    f1_score
)

import pandas as pd

In [ ]:
from sklearn.metrics import f1_score
import numpy as np


# ==========================================
# Official Favg2
# ==========================================

def compute_favg2(labels, predictions, targets):

    labels = np.asarray(labels)
    predictions = np.asarray(predictions)
    targets = np.asarray(targets)

    target_scores = []

    for target in np.unique(targets):

        mask = targets == target

        target_labels = labels[mask]
        target_predictions = predictions[mask]

        favor_f1 = f1_score(
            target_labels,
            target_predictions,
            labels=[0],
            average="macro",
            zero_division=0
        )

        against_f1 = f1_score(
            target_labels,
            target_predictions,
            labels=[1],
            average="macro",
            zero_division=0
        )

        target_scores.append(
            (favor_f1 + against_f1) / 2
        )

    return float(np.mean(target_scores))


# ==========================================
# Official Favg3
# ==========================================

def compute_favg3(labels, predictions, targets):

    labels = np.asarray(labels)
    predictions = np.asarray(predictions)
    targets = np.asarray(targets)

    target_scores = []

    for target in np.unique(targets):

        mask = targets == target

        target_labels = labels[mask]
        target_predictions = predictions[mask]

        score = f1_score(
            target_labels,
            target_predictions,
            average="macro",
            zero_division=0
        )

        target_scores.append(score)

    return float(np.mean(target_scores))

In [ ]:
#For Dev Set only

label2id = {
    "Favor": 0,
    "Against": 1,
    "None": 2
}

gold = eval_df["stance"].map(label2id).tolist()


In [ ]:
#For Dev Set only
results = []

for variant, prediction_list in all_variant_predictions.items():

    preds = [

        label2id[p]

        for p in prediction_list

    ]

    accuracy = accuracy_score(
        gold,
        preds
    )

    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(

        gold,
        preds,
        average="macro",
        zero_division=0

    )

    favg2 = compute_favg2(
        gold,
        preds,
        eval_df["target"]
    )

    favg3 = compute_favg3(
        gold,
        preds,
        eval_df["target"]
    )

    results.append({

        "Variant": variant,

        "Accuracy": accuracy,

        "Macro F1": macro_f1,

        "Favg2": favg2,

        "Favg3": favg3

    })


In [ ]:
#For Dev Set Only
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(

    "Favg2",

    ascending=False

)

display(results_df)

results_df.to_csv(

    "prompt_variant_results.csv",

    index=False

)


In [ ]:

import zipfile

# ==========================================
# Save All Variants as Competition Submission
# ==========================================

VALID_LABELS = {"Favor", "Against", "None"}

for variant, predictions in all_variant_predictions.items():

    if len(predictions) != len(eval_df):
        raise ValueError(
            f"{variant}: Expected {len(eval_df)} predictions, "
            f"got {len(predictions)}"
        )

    invalid = [
        p for p in predictions
        if p not in VALID_LABELS
    ]

    if invalid:
        raise ValueError(
            f"{variant}: Invalid labels found: {set(invalid)}"
        )

    txt_file = f"{variant}_submission.txt"
    zip_file = f"{variant}_submission.zip"

    # Write one label per line
    with open(txt_file, "w", encoding="utf-8") as f:

        for label in predictions:
            f.write(label + "\n")

    # Compress
    with zipfile.ZipFile(
        zip_file,
        "w",
        compression=zipfile.ZIP_DEFLATED
    ) as z:

        z.write(txt_file)

    print("=" * 60)
    print(f"{variant} submission created")
    print(f"Predictions : {len(predictions)}")
    print(f"TXT : {txt_file}")
    print(f"ZIP : {zip_file}")
